<a href="https://colab.research.google.com/github/satyajeetprabhu/beat-this-carnatic/blob/main/notebooks/Eval_FT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluate Beat This! Finetuned (BeatThis-FT)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Set Up Environment

In [2]:
!pip install git+https://github.com/satyajeetprabhu/beat-this-carnatic.git
!pip install git+https://github.com/mir-dataset-loaders/mirdata.git
!pip install mir_eval

  Cloning https://github.com/satyajeetprabhu/beat-this-carnatic.git to /tmp/pip-req-build-8txu8zei
  Running command git clone --filter=blob:none --quiet https://github.com/satyajeetprabhu/beat-this-carnatic.git /tmp/pip-req-build-8txu8zei
  Resolved https://github.com/satyajeetprabhu/beat-this-carnatic.git to commit 8d6bb71e68757c42e764579478c2f76ecd4d0ff8
  Preparing metadata (setup.py) ... done
  Created wheel for beat-this: filename=beat_this-0.1-py3-none-any.whl size=43288 sha256=06b689ec6f795bdc9274028d43781270c8f1e244d9ada7ff365b603348ddbbb1
  Stored in directory: /tmp/pip-ephem-wheel-cache-cmlesej4/wheels/13/d6/13/719341eac270ad2cc837dd1137b7136c9308e0a136cc1c8fe3
Successfully built beat-this
  Cloning https://github.com/mir-dataset-loaders/mirdata.git to /tmp/pip-req-build-bbw0hkv4
  Running command git clone --filter=blob:none --quiet https://github.com/mir-dataset-loaders/mirdata.git /tmp/pip-req-build-bbw0hkv4
  Resolved https://github.com/mir-dataset-loaders/mirdata.git to

In [3]:
# load the Python class for beat tracking
from beat_this.inference import File2File
from beat_this.inference import File2Beats
from beat_this.utils import save_beat_tsv
import os
import glob
import torch
from pathlib import PosixPath
# Add PosixPath to the safe globals
# Allow numpy._core.multiarray.scalar to be unpickled
import numpy as np
torch.serialization.add_safe_globals([np._core.multiarray.scalar, PosixPath])
import pandas as pd
import mirdata

## Load Carnatic Data

In [4]:
dataset_path = '/content/drive/MyDrive/Datasets/CMR'
carn = mirdata.initialize('compmusic_carnatic_rhythm', version='full_dataset_1.0', data_home=dataset_path)
carn.download(['index'])
carn_tracks = carn.load_tracks()
audio_folder = os.path.join(dataset_path, 'CMR_full_dataset_1.0', 'audio')

80.0kB [00:01, 73.2kB/s]                            
    the research-related use you will give to the dataset. Once the access is granted (it may take, at most, one day or two), please download 
    the dataset with the provided Zenodo link and uncompress the two zip files: CMR_full_dataset_1.0.zip and CMR_subset_1.0.zip. You don't need 
    to re-arrange or change the folder structure of these two versions, the dataloader is designed to work with the provided file organization. 
    Therefore, simply uncompress and store the datasets to a desired location, and use such location to initialize the dataset as follows: 
    
    compmusic_carnatic_rhythm = mirdata.initialize("compmusic_carnatic_rhythm", data_home="/path/to/home/folder/of/dataset").
    


### Set Path

Assumes folder structure from Pre-Processing and Finetuning notebooks

In [5]:
path = '/content/drive/MyDrive/Beat_This_CMR'

## SET VALUES HERE
    
trainfold in [1, 2].  
seed in [42, 52, 62].  

In [41]:
trainfold = 2
seed = 62

In [42]:
checkpoint_path = f'{path}/pretrained/bt-cmr-fold{trainfold}-finetune-50 S{seed}.ckpt'

In [43]:
# Output folder
output_path = os.path.join(path, 'beat-this-carnatic', 'output', 'predictions', 'ft')
output_folder = f'{output_path}/trainfold{trainfold}_seed{seed}'
os.makedirs(output_folder, exist_ok=True)

### Create Splits

In [44]:
testfold = 3 - trainfold

In [45]:
splits_csv_path = f'{path}/beat-this-carnatic/cmr_splits.csv'
splits_df = pd.read_csv(splits_csv_path, dtype={'track_id': str})

In [46]:
testfold_tracks = splits_df[splits_df['Fold'] == testfold]['track_id'].tolist()

## Predict on finetuned model checkpoints

In [47]:
file2beats = File2Beats(checkpoint_path=checkpoint_path, device="cuda", dbn=False)

# Process each .wav file
for file_id in testfold_tracks:
    audio_path = carn_tracks[file_id].audio_path

    # Get beat and downbeat predictions
    beats, downbeats = file2beats(audio_path)

    # Save as .beats file
    out_path = os.path.join(output_folder, f"{file_id}.beats")
    save_beat_tsv(beats, downbeats, out_path)

    print(f"Processed {file_id} -> {out_path}")

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Processed 10001 -> /content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic/output/predictions/ft/trainfold2_seed62/10001.beats
Processed 10002 -> /content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic/output/predictions/ft/trainfold2_seed62/10002.beats
Processed 10005 -> /content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic/output/predictions/ft/trainfold2_seed62/10005.beats
Processed 10009 -> /content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic/output/predictions/ft/trainfold2_seed62/10009.beats
Processed 10011 -> /content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic/output/predictions/ft/trainfold2_seed62/10011.beats
Processed 10014 -> /content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic/output/predictions/ft/trainfold2_seed62/10014.beats
Processed 10015 -> /content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic/output/predictions/ft/trainfold2_seed62/10015.beats
Processed 10016 -> /content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic/output/predictions/ft/trainfold2_see

## Evaluate models and Save results

Run this once after generating all the predictions

In [51]:
%cd {path}/beat-this-carnatic

/content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic


In [52]:
!python launch_scripts/eval_beat-this.py --data-home $dataset_path --mode 'ft'

    the research-related use you will give to the dataset. Once the access is granted (it may take, at most, one day or two), please download 
    the dataset with the provided Zenodo link and uncompress the two zip files: CMR_full_dataset_1.0.zip and CMR_subset_1.0.zip. You don't need 
    to re-arrange or change the folder structure of these two versions, the dataloader is designed to work with the provided file organization. 
    Therefore, simply uncompress and store the datasets to a desired location, and use such location to initialize the dataset as follows: 
    
    compmusic_carnatic_rhythm = mirdata.initialize("compmusic_carnatic_rhythm", data_home="/path/to/home/folder/of/dataset").
    
Evaluating Beat This version - ft
Evaluating model: trainfold1_seed42
Processing 88/88 tracks
Results saved to /content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic/output/results/beat-this_ft_trainfold1_seed42.csv
Evaluating model: trainfold1_seed52
Processing 88/88 tracks
Results saved 